In [1]:
import os
import json
import pandas as pd
import numpy as np
import cv2
from scipy.signal.windows import gaussian as scipy_gaussian

import matplotlib.pyplot as plt

In [2]:
# 1. Paramètres
CSV_PATH    = "label_annotated.csv"   # votre export CSV de VIA
IMG_DIR     = "dataset_project_iapr2025/train"                # dossier de vos .JPG
OUT_DIR     = "density_maps/"                                 # dossier où sauver les .npz
SIGMA       = 150                                             # écart-type de la gaussienne
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Crème_brûlée",
    "Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]
# 2. Mapping nom → indice
class2idx = {name: i for i, name in enumerate(CLASS_NAMES)}

In [4]:
# 3. Précompute the Gaussian kernel
kern_size = int(6 * SIGMA) | 1   # kernel size, ensure odd
g1d = scipy_gaussian(kern_size, SIGMA)
kernel2d = np.outer(g1d, g1d)
kernel2d /= kernel2d.sum()       # normalize sum to 1
half_k = kern_size // 2

In [5]:
# 4. Charger le CSV
df = pd.read_csv(CSV_PATH)

FileNotFoundError: [Errno 2] No such file or directory: 'label_annotated.csv'

In [19]:
# 5. Créer le dossier de sortie
os.makedirs(OUT_DIR, exist_ok=True)

In [20]:
# 6. Boucle sur chaque image
for fname, group in df.groupby("filename"):
    # 6.1 Lire l'image pour connaître ses dimensions
    img_path = os.path.join(IMG_DIR, fname)
    img = cv2.imread(img_path)
    if img is None:
        print(f"[ERROR] Impossible de lire l'image : {img_path}")
        continue
    H, W = img.shape[:2]
    
    # 6.2 Initialiser le tenseur de density maps
    density = np.zeros((H, W, len(CLASS_NAMES)), dtype=np.float32)
    
    # 6.3 Pour chaque point annoté, scatter-add le kernel
    for _, row in group.iterrows():
        shape_attr = json.loads(row["region_shape_attributes"])
        x = int(shape_attr["cx"])
        y = int(shape_attr["cy"])
        choco_type = json.loads(row["region_attributes"])["Choco_Type"]
        c = class2idx.get(choco_type, None)
        if c is None:
            continue
        
        # bounds for scatter-add
        y0 = max(0,   y - half_k)
        y1 = min(H,   y + half_k + 1)
        x0 = max(0,   x - half_k)
        x1 = min(W,   x + half_k + 1)
        ky0 = half_k - (y - y0)
        ky1 = half_k + (y1 - y)
        kx0 = half_k - (x - x0)
        kx1 = half_k + (x1 - x)
        
        density[y0:y1, x0:x1, c] += kernel2d[ky0:ky1, kx0:kx1]
    
    # 6.4 Conversion en float16 pour réduire la taille
    density = density.astype(np.float16)
    
    # 6.5 Sauvegarder au format compressé .npz
    out_path = os.path.join(OUT_DIR, fname.replace(".JPG", ".npz"))
    np.savez_compressed(out_path, density=density)
    
    print(f"Saved compressed density map for {fname} → {out_path}")


Saved compressed density map for L1000756.JPG → density_maps/L1000756.npz
Saved compressed density map for L1000763.JPG → density_maps/L1000763.npz
Saved compressed density map for L1000765.JPG → density_maps/L1000765.npz


KeyboardInterrupt: 

In [1]:
# exemple de chargement d'une density map et de l'image correspondante
npz = np.load("density_maps/L1000763.npz")
density = npz["density"]    # shape (H, W, n_classes)
img = cv2.imread("dataset_project_iapr2025/train/L1000763.JPG")[:,:,::-1]

# visualisation de la somme sur les canaux (densité totale)
total_density = density.sum(axis=-1)

fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].imshow(img); axes[0].set_title("Image originale"); axes[0].axis("off")
axes[1].imshow(total_density, cmap="jet"); axes[1].set_title("Density map (somme)"); axes[1].axis("off")
plt.show()

# vérifiez aussi la somme pour estimer le nombre d'objets
print("Nombre estimé d'objets sur l'image :", total_density.sum())


NameError: name 'np' is not defined

# Neural network :

In [3]:
import os
import json
import random
from glob import glob

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [4]:
# ─── 1. CONFIG ────────────────────────────────────────────────────────────────
DM_DIR      = "density_maps"
N_CLASSES = len(CLASS_NAMES)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH     = 4
EPOCHS    = 30
LR        = 1e-4
SEED      = 42

# Réglage du seed
#torch.manual_seed(SEED)
#np.random.seed(SEED)
#random.seed(SEED)

In [ ]:
import os, random, cv2, numpy as np, torch
from torch.utils.data import Dataset
"""
class ChocolateDataset(Dataset):
    def __init__(self, files, img_dir, dm_dir,
                 target_size=(600,400),
                 affine_params=None,
                 augment=True):
        
        #files         : liste de noms de fichiers .JPG
        #img_dir, dm_dir: dossiers images / .npz
        #target_size   : (H, W) taille en sortie
        #affine_params : dict avec
        #    'translate_frac': fraction max de W/H pour la translation
        #    'scale_range'   : tuple (min, max) pour le zoom
        #    'shear_deg'     : angle max de cisaillement en degrés
        #augment       : bool, si False on désactive la partie aléatoire
        
        self.files       = files
        self.img_dir     = img_dir
        self.dm_dir      = dm_dir
        self.Th, self.Tw = target_size
        self.ap = affine_params or {
            'translate_frac':0.1,
            'scale_range':(0.9,1.1),
            'shear_deg':10
        }
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        # 1) Lecture & resize
        img = cv2.imread(os.path.join(self.img_dir, fname))
        if img is None:
            raise FileNotFoundError(f"Image not found: {fname}")
        img = img[:,:,::-1].astype(np.float32)/255.0
        H0, W0 = img.shape[:2]
        img = cv2.resize(img, (self.Tw, self.Th), interpolation=cv2.INTER_AREA)

        dm = np.load(os.path.join(self.dm_dir, fname.replace(".JPG",".npz")))["density"].astype(np.float32)
        scale_area = (H0 * W0) / (self.Th * self.Tw)
        dm_chw = dm.transpose(2,0,1)
        dm_rs = np.zeros((dm_chw.shape[0], self.Th, self.Tw), dtype=np.float32)
        for c in range(dm_chw.shape[0]):
            patch = cv2.resize(dm_chw[c], (self.Tw, self.Th), interpolation=cv2.INTER_AREA)
            dm_rs[c] = patch * scale_area

        # 2) Tirage des paramètres affine uniquement si augment=True
        if self.augment:
            tx = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Tw
            ty = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Th
            sc = random.uniform(*self.ap['scale_range'])
            sh = random.uniform(-self.ap['shear_deg'], self.ap['shear_deg'])
            a, d = sc, sc
            b = np.tan(np.deg2rad(sh))
            c_ = 0.0
            M = np.array([[a, b, tx],
                          [c_, d, ty]], dtype=np.float32)
            det = a * d - b * c_

            img = cv2.warpAffine(img, M, (self.Tw, self.Th),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REFLECT)
            for c in range(dm_rs.shape[0]):
                dm_rs[c] = cv2.warpAffine(dm_rs[c], M, (self.Tw, self.Th),
                                          flags=cv2.INTER_LINEAR,
                                          borderMode=cv2.BORDER_REFLECT) * det

        # 3) Conversion en Tensors
        img_t = torch.from_numpy(img.transpose(2,0,1)).float()   # (3,H,W)
        dm_t  = torch.from_numpy(dm_rs).float()                   # (C,H,W)

        return img_t, dm_t
"""
class ChocolateDataset(Dataset):
    def __init__(self,
                 files, img_dir, dm_dir,
                 csv_path, class2idx,
                 target_size=(600,400),
                 affine_params=None,
                 augment=True):
        """
        + csv_path    : chemin vers le CSV VIA original
        + class2idx   : dict {classe_str → indice}
        """
        self.files       = files
        self.img_dir     = img_dir
        self.dm_dir      = dm_dir
        self.Th, self.Tw = target_size
        self.ap = affine_params or {
            'translate_frac':0.1,
            'scale_range':(0.9,1.1),
            'shear_deg':10
        }
        self.augment = augment

        # --- NOUVEAU : charger les annotations de points
        df = pd.read_csv(csv_path)
        # on s'assure que filename column matche vos files
        self.annots = {}
        for fname, grp in df.groupby("filename"):
            pts = []
            for _, row in grp.iterrows():
                sa = json.loads(row["region_shape_attributes"])
                ca = json.loads(row["region_attributes"])["Choco_Type"]
                if ca not in class2idx: 
                    continue
                pts.append((sa["cx"], sa["cy"], class2idx[ca]))
            self.annots[fname] = pts

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        # 1) Lecture & resize image
        img  = cv2.imread(os.path.join(self.img_dir, fname))
        img  = img[:,:,::-1].astype(np.float32)/255.0
        H0,W0 = img.shape[:2]
        img  = cv2.resize(img, (self.Tw, self.Th), interpolation=cv2.INTER_AREA)

        # 2) Lire & resize les density maps
        dm = np.load(os.path.join(self.dm_dir, fname.replace(".JPG",".npz")))["density"].astype(np.float32)
        scale_area = (H0 * W0) / (self.Th * self.Tw)
        dm_chw = dm.transpose(2,0,1)
        dm_rs  = np.zeros((dm_chw.shape[0], self.Th, self.Tw), dtype=np.float32)
        for c in range(dm_chw.shape[0]):
            patch     = cv2.resize(dm_chw[c], (self.Tw, self.Th), interpolation=cv2.INTER_AREA)
            dm_rs[c]  = patch * scale_area

        # 3) Générer le mask binaire d'impulsions (avant blur)
        mask = np.zeros((dm_chw.shape[0], self.Th, self.Tw), dtype=np.uint8)
        for (cx, cy, cidx) in self.annots.get(fname, []):
            # coordonnée redimensionnée
            x0 = cx * (self.Tw / W0)
            y0 = cy * (self.Th / H0)
            ix = int(round(x0))
            iy = int(round(y0))
            if 0 <= ix < self.Tw and 0 <= iy < self.Th:
                mask[cidx, iy, ix] = 1

        # 4) Application éventuelle des mêmes affines à img, dm_rs et mask
        if self.augment:
            tx = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Tw
            ty = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Th
            sc = random.uniform(*self.ap['scale_range'])
            sh = random.uniform(-self.ap['shear_deg'], self.ap['shear_deg'])
            a, d = sc, sc
            b     = np.tan(np.deg2rad(sh))
            c_    = 0.0
            M     = np.array([[a, b, tx],
                              [c_, d, ty]], dtype=np.float32)
            det   = a*d - b*c_

            img = cv2.warpAffine(img, M, (self.Tw,self.Th),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT,
                                 borderValue=0)

            for c in range(dm_rs.shape[0]):
                dm_rs[c] = cv2.warpAffine(dm_rs[c], M, (self.Tw, self.Th),
                                        flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_CONSTANT,
                                        borderValue=0
                                    ) * det
                                    
                mask[c] = cv2.warpAffine(mask[c], M, (self.Tw, self.Th),
                                        flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT,
                                        borderValue=0
                                    )

        # 5) Conversion en Tensors
        img_t   = torch.from_numpy(img.transpose(2,0,1)).float()   # (3,H,W)
        dm_t    = torch.from_numpy(dm_rs).float()                   # (C,H,W)
        mask_t  = torch.from_numpy(mask).float()                    # (C,H,W) 0/1

        return img_t, dm_t, mask_t



from glob import glob
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# 1) Récupère tous tes fichiers .JPG
all_files = [os.path.basename(p) for p in glob(os.path.join(IMG_DIR, "*.JPG"))]

# 2) Split train / val
train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)

# 3) Datasets
train_ds = ChocolateDataset(
    files=train_files,
    img_dir=IMG_DIR,
    dm_dir=DM_DIR,
    csv_path=CSV_PATH,           # <-- nouveau
    class2idx=class2idx,         # <-- nouveau
    target_size=(400,600),
    affine_params={'translate_frac':0.1,
                   'scale_range':(0.9,1.1),
                   'shear_deg':10},
    augment=True
)

val_ds = ChocolateDataset(
    files=val_files,
    img_dir=IMG_DIR,
    dm_dir=DM_DIR,
    csv_path=CSV_PATH,
    class2idx=class2idx,
    target_size=(400,600),
    affine_params={'translate_frac':0.1,
                   'scale_range':(0.9,1.1),
                   'shear_deg':10},
    augment=False
)
# 4) DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=0, pin_memory=False)


In [6]:
"""
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(3,   64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128,256)
        self.down4 = DoubleConv(256,512)
        self.pool  = nn.MaxPool2d(2)
        # Decoder
        self.up3   = nn.ConvTranspose2d(512,256,2,stride=2)
        self.conv3 = DoubleConv(512,256)
        self.up2   = nn.ConvTranspose2d(256,128,2,stride=2)
        self.conv2 = DoubleConv(256,128)
        self.up1   = nn.ConvTranspose2d(128,64, 2,stride=2)
        self.conv1 = DoubleConv(128,64)
        # Final conv + ReLU pour garantir non-négativité
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)
        self.final_relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))
        # Decoder
        u3 = self.up3(d4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        # Final layers
        x  = self.final_conv(u1)
        x  = self.final_relu(x)  # clamp non-négatif
        return x
"""
# ─── 1. PARAMÈTRES GAUSSIEN ─────────────────────────────────────────────────────
SIGMA       = 8   # comme défini précédemment mais images resized
kern_size   = (int(6 * SIGMA) | 1)   # taille impair ≃ 6σ
half_k      = kern_size // 2

# Génération du kernel 2D gaussien normalisé
x = np.arange(-half_k, half_k+1)
g1d = np.exp(-(x**2)/(2*SIGMA**2))
g1d /= g1d.sum()
kernel2d = np.outer(g1d, g1d).astype(np.float32)  # shape (kern_size, kern_size)
# Converti en tensor pour charger dans le module PyTorch
kernel_tensor = torch.from_numpy(kernel2d)[None, None, :, :]  # (1,1,k,k)

# ─── 2. DOUBLE CONV ─────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c,  out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

# ─── 3. UNET AVEC BLUR FIXE ─────────────────────────────────────────────────────
class UNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(3,   64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128,256)
        self.down4 = DoubleConv(256,512)
        self.pool  = nn.MaxPool2d(2)
        # Decoder
        self.up3   = nn.ConvTranspose2d(512,256,2,stride=2)
        self.conv3 = DoubleConv(512,256)
        self.up2   = nn.ConvTranspose2d(256,128,2,stride=2)
        self.conv2 = DoubleConv(256,128)
        self.up1   = nn.ConvTranspose2d(128,64,2,stride=2)
        self.conv1 = DoubleConv(128,64)

        # Tête d'impulsions (sans blur)
        self.impulse_head = nn.Conv2d(64, n_classes, kernel_size=1)
        self.impulse_relu = nn.ReLU(inplace=True)

        # Blur gaussien fixe en Conv2d groupé
        self.blur = nn.Conv2d(
            in_channels=n_classes,
            out_channels=n_classes,
            kernel_size=kern_size,
            padding=half_k,
            groups=n_classes,
            bias=False
        )
        # Charger et bloquer le kernel
        with torch.no_grad():
            self.blur.weight.copy_(kernel_tensor.repeat(n_classes,1,1,1))
        self.blur.weight.requires_grad_(False)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))
        # Decoder
        u3 = self.up3(d4); u3 = torch.cat([u3, d3], dim=1); u3 = self.conv3(u3)
        u2 = self.up2(u3); u2 = torch.cat([u2, d2], dim=1); u2 = self.conv2(u2)
        u1 = self.up1(u2); u1 = torch.cat([u1, d1], dim=1); u1 = self.conv1(u1)

        # Impulsions + ReLU
        impulses = self.impulse_head(u1)
        impulses = self.impulse_relu(impulses)
        # Blur fixe → density maps finales
        density = self.blur(impulses)

        return density, impulses
        
def init_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        # Kaiming He initialization for ReLU activations
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        # gamma=1, beta=0
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

model = UNet(N_CLASSES).to(DEVICE)
model.apply(init_weights)

UNet(
  (down1): DoubleConv(
    (net): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down2): DoubleConv(
    (net): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down3): DoubleConv(
    (net): Sequ

In [7]:
from torchsummary import summary
summary(model, (3, 512, 512), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 512, 512]           1,728
       BatchNorm2d-2          [1, 64, 512, 512]             128
              ReLU-3          [1, 64, 512, 512]               0
            Conv2d-4          [1, 64, 512, 512]          36,864
       BatchNorm2d-5          [1, 64, 512, 512]             128
              ReLU-6          [1, 64, 512, 512]               0
        DoubleConv-7          [1, 64, 512, 512]               0
         MaxPool2d-8          [1, 64, 256, 256]               0
            Conv2d-9         [1, 128, 256, 256]          73,728
      BatchNorm2d-10         [1, 128, 256, 256]             256
             ReLU-11         [1, 128, 256, 256]               0
           Conv2d-12         [1, 128, 256, 256]         147,456
      BatchNorm2d-13         [1, 128, 256, 256]             256
             ReLU-14         [1, 128, 2

In [8]:
# ─── 4. LOSS, OPTI, SCHEDULER ───────────────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True)

/opt/jlab-env/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [9]:
# ─── 5. FONCTIONS D’ENTRAÎNEMENT & ÉVAL ─────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss for multi-channel heatmaps (pixel-wise binary)
      FL(p,t) = -α t (1-p)^γ log(p) - (1-α)(1-t) p^γ log(1-p)
    where p = sigmoid(logit), t∈{0,1}.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean", eps: float = 1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: (B,C,H,W), targets: same shape, binary {0,1}
        p = torch.sigmoid(logits).clamp(self.eps, 1.0 - self.eps)
        # focal weights
        w = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        # pt = p if t==1 else (1-p)
        pt = p * targets + (1 - p) * (1 - targets)
        # focal loss
        loss = -w * (1 - pt).pow(self.gamma) * torch.log(pt)
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss
            

def train_epoch(loader, model,
                mse_criterion, focal_criterion,
                optimizer,
                lambda_focal=0.1,
                lambda_l1=1e-3):
    model.train()
    running_loss = 0.0

    for imgs, dms, mask in tqdm(loader, desc="Train", leave=False):
        imgs, dms, mask = imgs.to(DEVICE), dms.to(DEVICE), mask.to(DEVICE)

        # forward → density & logits
        density, logits = model(imgs)

        # compute each loss term
        loss_mse   = mse_criterion(density, dms)
        loss_focal = focal_criterion(logits, mask)
        loss_l1    = lambda_l1 * logits.abs().mean()

        # total loss
        loss = loss_mse + lambda_focal * loss_focal + loss_l1

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


def eval_epoch(loader, model,
               mse_criterion, focal_criterion,
               lambda_focal=0.1,
               lambda_l1=1e-3):
    model.eval()
    running_loss = 0.0
    all_true, all_pred = [], []
    visualized = True

    with torch.no_grad():
        for imgs, dms, mask in tqdm(loader, desc="Validation", leave=False):
            imgs, dms, mask = imgs.to(DEVICE), dms.to(DEVICE), mask.to(DEVICE)

            # forward → density & logits
            density, logits = model(imgs)

            # same loss as in train
            loss_mse   = mse_criterion(density, dms)
            loss_focal = focal_criterion(logits, mask)
            loss_l1    = lambda_l1 * logits.abs().mean()
            loss       = loss_mse + lambda_focal * loss_focal + loss_l1

            running_loss += loss.item() * imgs.size(0)

            # visualiser la première image du loader
            if visualized:
                img_np   = imgs[0].cpu().permute(1,2,0).numpy()
                gt_den   = dms[0].sum(dim=0).cpu().numpy()
                imp_sum  = torch.sigmoid(logits[0]).sum(dim=0).cpu().numpy()
                pred_den = density[0].sum(dim=0).cpu().numpy()

                fig, axes = plt.subplots(1,3,figsize=(18,6))
                axes[0].imshow(img_np)
                axes[0].imshow(gt_den,  cmap="jet", alpha=0.5)
                axes[0].set_title("GT : total density"); axes[0].axis("off")

                axes[1].imshow(img_np)
                axes[1].imshow(imp_sum, cmap="hot", alpha=0.5)
                axes[1].set_title("Impulsions");     axes[1].axis("off")

                axes[2].imshow(img_np)
                axes[2].imshow(pred_den,cmap="jet", alpha=0.5)
                axes[2].set_title("Prédiction : density"); axes[2].axis("off")

                plt.tight_layout()
                plt.show()

    loss = running_loss / len(loader.dataset)
    return loss

In [ ]:
# ─── 6. BOUCLE PRINCIPALE ───────────────────────────────────────────────────────
best_val_loss = float('inf')
mse_criterion   = nn.MSELoss(reduction="sum")
focal_criterion = FocalLoss(alpha=0.25, gamma=2.0, reduction="mean")
lambda_focal         = 0.5
lambda_l1            = 1e-3

for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(train_loader, model,
                          mse_criterion, focal_criterion,
                          optimizer, lambda_focal, lambda_l1)
    val_loss = eval_epoch(
                          val_loader, model,
                          mse_criterion, focal_criterion,
                          lambda_focal, lambda_l1)
    scheduler.step(val_loss)

    print(f"Époch {epoch:02d} | "
          f"Train Loss: {tr_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | ")
          #f"MAE: {val_mae.mean():.3f} | "
          #f"F1_macro: {val_f1_macro:.3f}")

    # optionnel : afficher le F1 par classe
    #for cname, f1 in zip(CLASS_NAMES, val_f1_pc):
        #print(f"  F1 {cname:<15s}: {f1:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_choco_unet.pth")
        print("→ Modèle sauvegardé!")

Train:  36%|███▌      | 13/36 [01:39<02:55,  7.63s/it]

In [12]:
# … après avoir chargé votre modèle et instancié test_loader …

records = []
with torch.no_grad():
    for fnames, imgs in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs)  # shape (B, C, H, W), déjà ≥ 0 grâce à final_relu

        # Comptage par intégrale de chaque density-map
        counts = (
            preds
            .sum(dim=(2,3))         # somme spatiale → (B, C)
            .round()                # arrondi au plus proche
            .long()                 # tensor int64
            .cpu()
            .numpy()                # passage en numpy
        )

        for fname, cnt in zip(fnames, counts):
            # extraction de l'ID numérique (cf. code précédent)
            base   = os.path.splitext(fname)[0]
            digits = ''.join(filter(str.isdigit, base))
            id_num = int(digits)

            row = {"id": id_num}
            for i, cls in enumerate(CLASS_NAMES):
                row[cls] = int(cnt[i])
            records.append(row)

# Construction et sauvegarde du CSV
df = pd.DataFrame(records)
df = df[["id"] + CLASS_NAMES]
df.to_csv("submission.csv", index=False)
print(df.head())


        id  Jelly_White  Jelly_Milk  Jelly_Black  Amandina  Crème_brûlée  \
0  1000757            0          49            0         3             0   
1  1000758           80          96            0         0             0   
2  1000759           40          73            0         0             0   
3  1000760          152          77            0        35             0   
4  1000761           77          75            0         0             0   

   Triangolo  Tentation_noir  Comtesse  Noblesse  Noir_authentique  \
0          0               0       492        38               821   
1          0               0       224       269               655   
2          0               0       248       161               644   
3          0               0       359       151               674   
4          0               0       440        90               742   

   Passion_au_lait  Arabia  Stracciatella  
0                0       0              0  
1                0       0        